# Tutorial 6: Amplitude Estimation Variants

This notebook compares four QAE variants in qufin:
1. **Canonical QAE** (`CanonicalAmplitudeEstimation`) -- QPE-based, requires controlled-Grover and QFT
2. **IQAE** (`IterativeAmplitudeEstimation`) -- Iterative, no QFT, adaptive schedule
3. **MLAE** (`MaximumLikelihoodAmplitudeEstimation`) -- Maximum likelihood, fixed-schedule measurements
4. **FQAE** (`FaithfulAmplitudeEstimation`) -- Faithful AE, low circuit depth

qufin ships six QAE algorithms in total (the four above plus MRQAE and QMC).

**References**: Brassard et al. (2002), Grinko et al. (2021), Suzuki et al. (2020), Giurgica-Tiron et al. (2022).

In [ ]:
import numpy as np
import time
np.random.seed(42)

## 1. Common Problem Setup

We estimate a known amplitude $a = \sin^2\theta$ with all four QAE variants so
their accuracy and cost can be compared directly against the exact value.

In [ ]:
from qiskit.circuit import QuantumCircuit

from qufin.options.amplitude_estimation.estimation_problem import EstimationProblem
from qufin.options.classical.black_scholes import call_price
from qufin.backends.qiskit_backend import QiskitAerBackend

S, K, sigma, r, T = 100, 105, 0.2, 0.05, 1.0
bs_ref = call_price(s=S, k=K, sigma=sigma, r=r, T=T)

# All four variants estimate the same amplitude a = sin^2(theta) on a gate-based
# Bernoulli oracle A|0> = cos(theta)|0> + sin(theta)|1>. Using an invertible
# (gate-based) state preparation keeps the Grover operator well-defined.
theta = np.pi / 5
true_a = np.sin(theta) ** 2
oracle = QuantumCircuit(1)
oracle.ry(2 * theta, 0)
problem = EstimationProblem(state_preparation=oracle, objective_qubits=[0], n_qubits=1)
backend = QiskitAerBackend(method="automatic", seed=42)

print(f"Black-Scholes reference: ${bs_ref:.4f}")
print(f"Target amplitude a = sin^2(theta): {true_a:.4f}")

## 2. Canonical QAE (QPE-Based)

In [ ]:
from qufin.options.amplitude_estimation.canonical import (
    CanonicalAmplitudeEstimation, CanonicalQAEConfig,
)

t0 = time.time()
canonical = CanonicalAmplitudeEstimation(
    problem, CanonicalQAEConfig(n_eval_qubits=5), backend,
)
canonical_result = canonical.estimate()
t_canonical = time.time() - t0

print(f"Canonical QAE: {canonical_result.estimate:.4f}  (error={abs(canonical_result.estimate - true_a):.4f}, time={t_canonical:.2f}s)")

## 3. Iterative QAE (IQAE)

In [ ]:
from qufin.options.amplitude_estimation.iqae import (
    IQAEConfig, IterativeAmplitudeEstimation,
)

t0 = time.time()
iqae = IterativeAmplitudeEstimation(
    problem, IQAEConfig(epsilon_target=0.01, alpha=0.05, shots_per_round=2048), backend,
)
iqae_result = iqae.estimate()
t_iqae = time.time() - t0

print(f"IQAE:          {iqae_result.estimate:.4f}  (error={abs(iqae_result.estimate - true_a):.4f}, time={t_iqae:.2f}s)")

## 4. Maximum Likelihood QAE (MLAE)

In [ ]:
from qufin.options.amplitude_estimation.mlae import (
    MLAEConfig, MaximumLikelihoodAmplitudeEstimation,
)

t0 = time.time()
mlae = MaximumLikelihoodAmplitudeEstimation(
    problem, MLAEConfig(evaluation_schedule=[0, 1, 2, 4, 8], n_shots_per_round=1024), backend,
)
mlae_result = mlae.estimate()
t_mlae = time.time() - t0

print(f"MLAE:          {mlae_result.estimate:.4f}  (error={abs(mlae_result.estimate - true_a):.4f}, time={t_mlae:.2f}s)")

## 5. Fourier QAE (FQAE)

In [ ]:
from qufin.options.amplitude_estimation.fqae import (
    FQAEConfig, FaithfulAmplitudeEstimation,
)

t0 = time.time()
fqae = FaithfulAmplitudeEstimation(
    problem, FQAEConfig(max_depth=8, n_shots_per_round=2048), backend,
)
fqae_result = fqae.estimate()
t_fqae = time.time() - t0

print(f"FQAE:          {fqae_result.estimate:.4f}  (error={abs(fqae_result.estimate - true_a):.4f}, time={t_fqae:.2f}s)")

## 6. Comparison Table

In [ ]:
results = [
    ("Canonical", canonical_result.estimate, t_canonical),
    ("IQAE", iqae_result.estimate, t_iqae),
    ("MLAE", mlae_result.estimate, t_mlae),
    ("FQAE", fqae_result.estimate, t_fqae),
]

print(f"{'Method':<12} {'Estimate':>9} {'Error':>8} {'Time (s)':>10}")
print("-" * 42)
print(f"{'True a':<12} {true_a:>9.4f} {0.0:>8.4f} {'--':>10}")
for name, est, t in results:
    print(f"{name:<12} {est:>9.4f} {abs(est - true_a):>8.4f} {t:>10.2f}")

## Summary

| Method | QPE Required | Circuit Depth | Key Property |
|:-------|:-------------|:-------------|:-------------|
| Canonical | Yes | Deep (QFT + c-Grover) | Optimal query complexity |
| IQAE | No | Moderate (adaptive) | Near-optimal, no QFT |
| MLAE | No | Moderate (fixed schedule) | Robust to noise |
| FQAE | No | Shallow | Best for near-term hardware |

**Next**: Tutorial 07 applies QAE to risk management (VaR/CVaR).